In [12]:
import os
from tqdm import tqdm 

from pathlib import Path
from lightning import Trainer
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scvi
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
import torch
from torch.utils.data import TensorDataset, DataLoader
from torchdyn.core import NeuralODE
from tqdm.auto import tqdm 
from pathlib import Path 

from scRatio.datamodules.datamodule import AnnDataDataModule
from scRatio.models.flow_matching import ConditionalFlowMatchingWithScore

Data 

In [13]:
# AnnData path 
path = Path("../../project_folder/data/pbmc68k_oversampled/pbmc68k_0.4.h5ad")
dataloader = AnnDataDataModule(adata_path=path,
                               conditions=["treatment"], 
                               num_features=30,
                               train_batch_size=1024,
                               val_batch_size=0,
                               test_batch_size=0)

dataloader.setup()

train_loader = dataloader.train_dataloader()
batch = next(iter(train_loader))
batch

[tensor([[ 0.9363, -0.2060,  0.1485,  ..., -0.0962, -0.0553, -0.0503],
         [ 1.0409, -0.1355, -0.4573,  ..., -0.1466, -0.0485,  0.0494],
         [-0.5216,  0.3472,  1.0191,  ..., -0.0953,  0.0184,  0.1323],
         ...,
         [ 0.8485, -0.4422, -0.4957,  ..., -0.1479, -0.0162, -0.0389],
         [-0.8210, -0.3770, -0.2721,  ...,  0.8374, -0.0992, -0.3254],
         [ 0.6129,  0.8893, -1.0448,  ...,  0.0505,  0.7323,  0.1455]]),
 tensor([[0., 1.],
         [1., 0.],
         [1., 0.],
         ...,
         [1., 0.],
         [1., 0.],
         [0., 1.]])]

Model

In [14]:
# Initialize model 
sigma = 0
sigma_min = 0
lambda_t = lambda t: torch.sqrt((1 - (1 - sigma_min) * t) ** 2 + sigma * t * (1 - t))
lambda_sp_t = lambda t: (sigma * (1 - 2 * t) - 2 * (1 - sigma_min) * (1 - (1 - sigma_min) * t)) / 2

model = ConditionalFlowMatchingWithScore(input_dim=30,
                                         cond_dims=[2],
                                         hidden_dims=[1024, 1024, 1024],
                                         encoder_hidden_dims=[256],
                                         encoder_out_dim=256,
                                         encoder_out_dim_cond=50, 
                                         time_feature_dim=50, 
                                         lambda_t=lambda_t,
                                         lambda_sp_t=lambda_sp_t,
                                         betas=[0],
                                         lr = 1e-4,
                                         dropout = 0)

In [15]:
checkpoint_callback = ModelCheckpoint(
    monitor="train_loss",
    mode="min",
    save_last=True
)

trainer = Trainer(
    max_epochs=100,
    accelerator="cpu",
    callbacks=[checkpoint_callback]
)

trainer.fit(model, datamodule=dataloader)

/home/icb/alessandro.palma/miniconda3/envs/scRatio/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3.10 /home/icb/alessandro.palma/miniconda3/envs/scRat ...
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/icb/alessandro.palma/miniconda3/envs/scRatio/lib/python3.10/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/home/icb/alessandro.palma/miniconda3/envs/scRatio/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /ictstr01/home/icb/alessandro.palma/environment/scFM_density_estimation/notebooks/differential_abundance_analysis/lightning_logs/version_34042631/checkpoints exists and is not empty.


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ data_encoder  │ Encoder         │ 73.7 K │ train │     0 │
│ 1 │ cond_encoders │ ModuleList      │ 13.6 K │ train │     0 │
│ 2 │ vf_mlp        │ FlowMatchingMLP │  2.5 M │ train │     0 │
│ 3 │ score_mlp     │ FlowMatchingMLP │  2.5 M │ train │     0 │
└───┴───────────────┴─────────────────┴────────┴───────┴───────┘

Trainable params: 5.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.1 M                                                                                                
Total estimated model params size (MB): 20                                                                         
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/icb/alessandro.palma/miniconda3/envs/scRatio/lib/python3.10/site-packages/lightning/pytorch/utilities/data.py
:106: Total length of `list` across ranks is zero. Please make sure this was your intention.

/home/icb/alessandro.palma/miniconda3/envs/scRatio/lib/python3.10/site-packages/lightning/pytorch/trainer/connector
s/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=5` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=100` reached.


Evaluate ratio

In [18]:
x_ratio = dataloader.train_dataset.x[:100]
control = torch.tensor([1., 0.])
condition = torch.tensor([0., 1.])

ratios = model.estimate_log_density_ratio(x_ratio, 
                                          control.unsqueeze(0).repeat(x_ratio.shape[0], 1), 
                                          condition.unsqueeze(0).repeat(x_ratio.shape[0], 1), 
                                          condition.unsqueeze(0).repeat(x_ratio.shape[0], 1), 
                                          n_steps=2)

# Add ratio to AnnData and visualize

In [ ]:
adata.obs["log_ratios"] = log_ratio

In [ ]:
sc.pl.umap(adata, color="log_ratios", s=10, cmap="plasma", vmin=-5, vmax=5)

In [17]:
sc.pl.umap(adata, color="leiden", s=10)

In [ ]:
sc.pp.neighbors(adata, n_pcs=30)
sc.tl.umap(adata)

In [ ]:
# sc.pl.umap(adata, color="leiden")

In [ ]:
# sc.pp.neighbors(adata, n_pcs=40)
# sc.tl.umap(adata)

In [ ]:
# sc.pl.umap(adata, color="leiden")

In [ ]:
# sc.pl.pca_variance_ratio(
#     adata,
#     log=True,      # optional, helps visualize elbow
#     n_pcs=50
# )

In [ ]:
# adata = sc.read_h5ad("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/scRatio/deterministic/oversamp_0.05_0.h5ad")

In [ ]:
# sc.pl.umap(adata, color="log_ratios", s=10, cmap="plasma", vmin=-5, vmax=5)

In [ ]:
# adata = sc.read_h5ad("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/scRatio_sweep/dimensions_10_batch_size_512_scheduler_type_stochastic_sigma_0.1/oversamp_0.2.h5ad")

In [ ]:
# adata.obs